In [1]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/Shareddrives/Progetti/Progetto_MLDM')

    GIT_BRANCH = 'zone-test'
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone -b {GIT_BRANCH} https://github.com/SimoRinaldi/crop-spatial-classification.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git fetch origin && git checkout -B {GIT_BRANCH} origin/{GIT_BRANCH} && git reset --hard origin/{GIT_BRANCH} && git clean -fd
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato! Branch attivo su Colab: ", GIT_BRANCH)
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = DATA_DIR / 'models'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")
     


Mounted at /content/drive
Cloning into '/content/crop-spatial-classification'...
remote: Enumerating objects: 931, done.
remote: Counting objects: 100% (349/349), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 931 (delta 277), reused 242 (delta 198), pack-reused 582 (from 1)
Receiving objects: 100% (931/931), 197.34 MiB | 27.13 MiB/s, done.
Resolving deltas: 100% (651/651), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 58.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 12.4 MB/s eta 0:00:00
Setup ambiente Colab completato! Branch attivo su Colab:  zone-test
✅ Collegamento ai dati riuscito! Cartella raw: /content/drive/Shareddrives/Progetti/Progetto_MLDM/data/raw


# 04 - Addestramento modelli con ricerca degli iperparametri migliori
- Addestramento dei modelli Random Forest, ExtraTrees e XGBoost
- Dataset di addestramento relativo all'area della Capitanata.

#### Caricamento del dataset e selezione delle feature predittive

In [2]:
import json
import pandas as pd

from src.feature_engineering import FEATURE_LIST

dataset_path = PROCESSED_DIR / 'dataset' / 'dataset.parquet'
legend_path = PROCESSED_DIR / 'legend.json'

dataset = pd.read_parquet(dataset_path)

# esclusione delle features non utili ai fini dell'addestramento
features = [f for f in dataset.columns if f in FEATURE_LIST]

X = dataset[features]
y = dataset["Ground_Truth"].astype(int)

# caricamento legenda delle colture
legend = {}
if legend_path.exists():
    with open(legend_path, "r", encoding="utf-8") as f:
        legend = json.load(f)

print(f"[Capitanata]\nDataset di addestramento:")
print(f"  • Records: {X.shape[0]}")
print(f"  • Features: {X.shape[1]}")
print(f"  • Classi: {y.nunique()}")

[Capitanata]
Dataset di addestramento:
  • Records: 22500
  • Features: 51
  • Classi: 17


#### RandomForest

In [3]:
import time
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
                                                                                                           
extra_trees_dir = MODELS_DIR / "random_forest"   
extra_trees_dir.mkdir(parents=True, exist_ok=True)
model_path = extra_trees_dir / "random_forest_crop_model.joblib"   

if model_path.exists():                                                                                                                         
    print(f"Modello RandomForestClassifier trovato su disco: {model_path.name}")                                                                                  
    print("Caricamento del modello completato!")                                                                                                
    payload = joblib.load(model_path)                                                                                                     
    best_model = payload["model"]                                                                                                         
    best_params = payload.get("best_params")                                                                                              
    best_cv_f1 = payload.get("best_cv_f1_macro")   
    cv_results = payload.get("cv_results")                                                                                       
    print(f"  • Parametri migliori: {best_params}")                                                                                             
    print(f"  • F1-Macro (CV):      {best_cv_f1:.4f}")                                                                                          
else:                                                                                                                                           
    print("Modello non trovato. Inizio addestramento RandomForestClassifier (GridSearchCV)...")                                                                        
    model = RandomForestClassifier(random_state=42, max_features="sqrt")                                                                        
    param_grid = {                                                                                                                              
        "n_estimators": [100, 300, 500, 700],                                                                                                   
        "max_depth": [None, 25],                                                                                                                
        "min_samples_split": [2],                                                                                                               
        "min_samples_leaf": [1],                                                                                                                
        "class_weight": ["balanced_subsample"],                                                                                                 
    }                                                                                                                                           
    grid_search = GridSearchCV(                                                                                                                 
        estimator=model,                                                                                                                        
        param_grid=param_grid,                                                                                                                  
        scoring=["accuracy", "f1_macro"],                                                                                                       
        refit="f1_macro",                                                                                                                       
        cv=5,                                                                                                                                   
        n_jobs=-1,                                                                                                                              
        verbose=1,                                                                                                                              
    )

    start_time = time.time()                                                                                                                    
    grid_search.fit(X, y)                                                                                                           
    elapsed = time.time() - start_time                                                                                                          
    print(f"Addestramento completato in {elapsed:.2f} secondi.")

    best_model = grid_search.best_estimator_                                                                                                    
    best_params = grid_search.best_params_                                                                                                      
    best_cv_f1 = grid_search.best_score_                                                                                                        
                                                                                                                                                
    # salvataggio del modello su disco                                                                                                          
    payload = {                                                                                                                                 
        "model": best_model,                                                                                                                    
        "best_params": best_params,                                                                                                             
        "best_cv_f1_macro": best_cv_f1,                                                                                                         
        "features": list(X.columns),                                                                                                      
        "n_features": X.shape[1],                                                                                                         
        "threshold": None,                                                                                                                      
        "training_time_seconds": elapsed,    
        "cv_results": grid_search.cv_results_,                                                                                                    
    }                                                                                                                                           
    joblib.dump(payload, model_path)                                                                                                            
    print(f"Modello salvato con successo in: {model_path.name}")                                                                                                    

Modello RandomForestClassifier trovato su disco: random_forest_crop_model.joblib
Caricamento del modello completato!
  • Parametri migliori: {'class_weight': 'balanced_subsample', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 700}
  • F1-Macro (CV):      0.5618


#### ExtraTrees

In [4]:
import time
import joblib
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV
                                                                                                           
extra_trees_dir = MODELS_DIR / "extra_trees"   
extra_trees_dir.mkdir(parents=True, exist_ok=True)
model_path = extra_trees_dir / "extra_trees_crop_model.joblib"   

if model_path.exists():                                                                                                                         
    print(f"Modello ExtraTreesClassifier trovato su disco: {model_path.name}")                                                                                  
    print("Caricamento del modello completato!")                                                                                                
    payload = joblib.load(model_path)                                                                                                     
    best_model = payload["model"]                                                                                                         
    best_params = payload.get("best_params")                                                                                              
    best_cv_f1 = payload.get("best_cv_f1_macro")   
    cv_results = payload.get("cv_results")                                                                                       
    print(f"  • Parametri migliori: {best_params}")                                                                                             
    print(f"  • F1-Macro (CV):      {best_cv_f1:.4f}")                                                                                          
else:                                                                                                                                           
    print("Modello non trovato. Inizio addestramento ExtraTreesClassifier (GridSearchCV)...")                                                                        
    model = ExtraTreesClassifier(random_state=42, max_features="sqrt")                                                                        
    param_grid = {                                                                                                                              
        "n_estimators": [100, 300, 500, 700],                                                                                                   
        "max_depth": [None, 25],                                                                                                                
        "min_samples_split": [2],                                                                                                               
        "min_samples_leaf": [1],                                                                                                                
        "class_weight": ["balanced_subsample"],                                                                                                 
    }                                                                                                                                           
    grid_search = GridSearchCV(                                                                                                                 
        estimator=model,                                                                                                                        
        param_grid=param_grid,                                                                                                                  
        scoring=["accuracy", "f1_macro"],                                                                                                       
        refit="f1_macro",                                                                                                                       
        cv=5,                                                                                                                                   
        n_jobs=-1,                                                                                                                              
        verbose=1,                                                                                                                              
    )

    start_time = time.time()                                                                                                                    
    grid_search.fit(X, y)                                                                                                           
    elapsed = time.time() - start_time                                                                                                          
    print(f"Addestramento completato in {elapsed:.2f} secondi.")

    best_model = grid_search.best_estimator_                                                                                                    
    best_params = grid_search.best_params_                                                                                                      
    best_cv_f1 = grid_search.best_score_                                                                                                        
                                                                                                                                                
    # salvataggio del modello su disco                                                                                                          
    payload = {                                                                                                                                 
        "model": best_model,                                                                                                                    
        "best_params": best_params,                                                                                                             
        "best_cv_f1_macro": best_cv_f1,                                                                                                         
        "features": list(X.columns),                                                                                                      
        "n_features": X.shape[1],                                                                                                         
        "threshold": None,                                                                                                                      
        "training_time_seconds": elapsed,    
        "cv_results": grid_search.cv_results_,                                                                                                    
    }                                                                                                                                           
    joblib.dump(payload, model_path)                                                                                                            
    print(f"Modello salvato con successo in: {model_path.name}")  

Modello ExtraTreesClassifier trovato su disco: extra_trees_crop_model.joblib
Caricamento del modello completato!
  • Parametri migliori: {'class_weight': 'balanced_subsample', 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 700}
  • F1-Macro (CV):      0.5648


#### XGBoost

In [2]:
import time
import joblib
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder                                               

xgb_dir = MODELS_DIR / "xgboost"   
xgb_dir.mkdir(parents=True, exist_ok=True)
model_path = xgb_dir / "xgboost_crop_model.joblib"   

if model_path.exists():                                                                                                                         
    print(f"Modello XGBClassifier trovato su disco: {model_path.name}")                                                                                  
    print("Caricamento del modello completato!")                                                                                                
    payload = joblib.load(model_path)                                                                                                     
    best_model = payload["model"]                                                                                                         
    best_params = payload.get("best_params")                                                                                              
    best_cv_f1 = payload.get("best_cv_f1_macro")   

    
    cv_results = payload.get("cv_results")                                                                                       
    print(f"  • Parametri migliori: {best_params}")                                                                                             
    print(f"  • F1-Macro (CV):      {best_cv_f1:.4f}")                                                                                          
else:                                                                                                                                           
    print("Modello non trovato. Inizio addestramento XGBClassifier (GridSearchCV)...")                                                                        
    model = XGBClassifier(                                                                       
        random_state=42,                                                                         
        colsample_bytree=0.8,                                                                    
        subsample=0.8,                                                                           
        eval_metric="mlogloss",                                                                  
    )      
    param_grid = {
        "n_estimators": [200, 400],
        "learning_rate": [0.05, 0.1],
        "max_depth": [6, 8],
        "min_child_weight": [1, 3],
    }
    grid_search = GridSearchCV(                                                                                                                 
        estimator=model,                                                                                                                        
        param_grid=param_grid,                                                                                                                  
        scoring=["accuracy", "f1_macro"],                                                                                                       
        refit="f1_macro",                                                                                                                       
        cv=5,                                                                                                                                   
        n_jobs=-1,                                                                                                                              
        verbose=1,                                                                                                                              
    )

    # codifica dei codici delle colture
    label_encoder = LabelEncoder()                                                               
    y_encoded = label_encoder.fit_transform(y)    

    start_time = time.time()                                                                                                                    
    grid_search.fit(X, y_encoded)                                                                                                           
    elapsed = time.time() - start_time                                                                                                          
    print(f"Addestramento completato in {elapsed:.2f} secondi.")

    best_model = grid_search.best_estimator_                                                                                                    
    best_params = grid_search.best_params_                                                                                                      
    best_cv_f1 = grid_search.best_score_                                                                                                        
                                                                                                                                                
    # salvataggio del modello su disco                                                                                                          
    payload = {                                                                                                                                 
        "model": best_model,                                                                                                                    
        "best_params": best_params,     
        "label_encoder": label_encoder,                                                                                                    
        "best_cv_f1_macro": best_cv_f1,                                                                                                         
        "features": list(X.columns),                                                                                                      
        "n_features": X.shape[1],                                                                                                         
        "threshold": None,                                                                                                                      
        "training_time_seconds": elapsed,    
        "cv_results": grid_search.cv_results_,                                                                                                    
    }                                                                                                                                           
    joblib.dump(payload, model_path)                                                                                                            
    print(f"Modello salvato con successo in: {model_path.name}")  


Modello XGBClassifier trovato su disco: xgboost_crop_model.joblib
Caricamento del modello completato!
  • Parametri migliori: {'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 400}
  • F1-Macro (CV):      0.5815
